In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Complementacao da CCJ do Senado

Notebook Colab para rodar um backfill especifico das notas taquigraficas da CCJ do Senado ate 2024.

A complementacao usa um `run_id` separado para preservar a coleta original. O caso de validacao e a reuniao `11176`, de `2023-03-29`: o metadado de notas indica `N`, mas a API textual e a pagina publica entregam texto.

In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

for name in ["raw", "checkpoints", "logs", "manifests", "processed"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("FALANDO_NELA_DATA_ROOT=", os.environ["FALANDO_NELA_DATA_ROOT"])

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## Parametros

A validacao curta usa somente `2023-03-29`, com `sample_limit=1`, para confirmar que a reuniao `11176` gera corpus textual apesar do indicador `N`. A complementacao completa fica protegida por uma flag.

In [ ]:
from datetime import datetime, timezone

DATA_INICIO_VALIDACAO = "2023-03-29"
DATA_FIM_VALIDACAO = "2023-03-29"
SAMPLE_LIMIT_VALIDACAO = 1
CODIGO_REUNIAO_VALIDACAO = "11176"

DATA_INICIO_COMPLEMENTO = "2011-05-18"
DATA_FIM_COMPLEMENTO = "2024-12-31"

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID_VALIDACAO = f"colab-senado-ccj-complemento-validacao-{RUN_STAMP}"
RUN_ID_COMPLEMENTO = "prod-senado-ccj-complemento-ate-2024"

print("RUN_ID_VALIDACAO=", RUN_ID_VALIDACAO)
print("RUN_ID_COMPLEMENTO=", RUN_ID_COMPLEMENTO, "# fixo para permitir retomada com --resume")

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

MODULE = "coleta.senado.ccj_notas.collect"


def run_collector(run_id, data_inicio, data_fim, sample=False, sample_limit=None, resume=True):
    cmd = [
        sys.executable,
        "-m",
        MODULE,
        "--mode",
        "prod",
        "--run-id",
        run_id,
        "--data-inicio",
        data_inicio,
        "--data-fim",
        data_fim,
    ]
    if resume:
        cmd.append("--resume")
    if sample:
        cmd.append("--sample")
    else:
        cmd.append("--no-sample")
    if sample_limit is not None:
        cmd.extend(["--sample-limit", str(sample_limit)])

    print("Rodando:", " ".join(cmd))
    output_lines = []
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line.rstrip("
"))

    returncode = process.wait()
    manifest_path = manifest_from_output(output_lines, run_id)
    if returncode != 0:
        print(f"Coletor terminou com returncode={returncode}; veja logs/autosave.")
    return manifest_path


def manifest_from_output(output_lines, run_id):
    for line in reversed(output_lines):
        candidate = line.strip()
        if candidate.endswith(".json"):
            path = Path(candidate)
            if path.exists():
                return path

    manifest_path = DATA_ROOT / "manifests" / f"{run_id}.json"
    autosave_path = DATA_ROOT / "manifests" / f"{run_id}.autosave.json"
    if manifest_path.exists():
        return manifest_path
    if autosave_path.exists():
        return autosave_path
    return None


def show_manifest(manifest_path):
    if manifest_path is None:
        print("Manifest nao foi informado pelo coletor.")
        return None

    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        print("Manifest nao encontrado:", manifest_path)
        return None

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    keys = [
        "run_id",
        "source",
        "dataset",
        "mode",
        "sample",
        "sample_limit",
        "status",
        "errors",
        "ccj_codigo",
        "reunioes_processadas",
        "record_counts",
        "partition_counts",
        "processed_records_loaded",
        "log_path",
        "autosave_path",
    ]
    resumo = {key: manifest.get(key) for key in keys}
    print(json.dumps(resumo, ensure_ascii=False, indent=2))

    log_path = Path(manifest["log_path"])
    if log_path.exists():
        print("
Ultimas linhas do log:")
        print("
".join(log_path.read_text(encoding="utf-8").splitlines()[-10:]))
    return manifest


def tail_log_for_run(run_id, n=30):
    log_path = DATA_ROOT / "logs" / f"{run_id}.jsonl"
    print("log_path=", log_path)
    if not log_path.exists():
        print("Log ainda nao encontrado.")
        return
    print("
".join(log_path.read_text(encoding="utf-8").splitlines()[-n:]))

## Validacao curta

Esta celula confirma que o caminho de complementacao esta ativo. O resultado esperado e um registro `notas_taquigraficas` para `reuniao:11176:notas_taquigraficas` em `ano=2023/mes=03`.

In [ ]:
manifest_validacao_path = run_collector(
    RUN_ID_VALIDACAO,
    DATA_INICIO_VALIDACAO,
    DATA_FIM_VALIDACAO,
    sample=False,
    sample_limit=SAMPLE_LIMIT_VALIDACAO,
    resume=True,
)
manifest_validacao = show_manifest(manifest_validacao_path)

In [ ]:
from datetime import date


def inspect_jsonl(path, max_rows=5):
    path = Path(path)
    print("
Arquivo:", path)
    if not path.exists():
        print("Nao encontrado")
        return []

    lines = path.read_text(encoding="utf-8").splitlines()
    print("linhas=", len(lines), "bytes=", path.stat().st_size)
    records = [json.loads(line) for line in lines]
    for record in records[:max_rows]:
        payload = record.get("payload", {})
        texto = payload.get("texto") or payload.get("TextoIntegral") or ""
        print({
            "record_type": record.get("record_type"),
            "partition": record.get("partition"),
            "source_id": record.get("source_id"),
            "metodo_obtencao": payload.get("metodo_obtencao") if isinstance(payload, dict) else None,
            "texto_status": payload.get("texto_status") if isinstance(payload, dict) else None,
            "codigo_reuniao": payload.get("codigo_reuniao") if isinstance(payload, dict) else None,
            "texto_chars": len(texto),
        })
    return records


validation_start = date.fromisoformat(DATA_INICIO_VALIDACAO)
metadata_path = DATA_ROOT / "raw" / "senado" / "ccj_notas" / "metadata" / f"{RUN_ID_VALIDACAO}.jsonl"
corpus_path = (
    DATA_ROOT
    / "raw"
    / "senado"
    / "ccj_notas"
    / f"ano={validation_start.year:04d}"
    / f"mes={validation_start.month:02d}"
    / f"{RUN_ID_VALIDACAO}.jsonl"
)

metadata_records = inspect_jsonl(metadata_path, max_rows=8)
corpus_records = inspect_jsonl(corpus_path, max_rows=8)

matches = [
    record for record in corpus_records
    if record.get("source_id") == f"reuniao:{CODIGO_REUNIAO_VALIDACAO}:notas_taquigraficas"
]
print("
Registros da reuniao", CODIGO_REUNIAO_VALIDACAO, "=", len(matches))
if matches:
    payload = matches[0]["payload"]
    print({
        "metodo_obtencao": payload.get("metodo_obtencao"),
        "texto_status": payload.get("texto_status"),
        "texto_chars": len(payload.get("texto") or ""),
        "tentativas": [item.get("metodo_obtencao") for item in payload.get("tentativas_texto", [])],
    })

## Complementacao ate 2024

Esta celula roda o backfill completo em modo producao, com `run_id` fixo e `--resume`. A execucao e separada da coleta original para facilitar auditoria e consolidacao posterior.

In [ ]:
EXECUTAR_COMPLEMENTO = False
RUN_ID_MADRUGADA = RUN_ID_COMPLEMENTO

if EXECUTAR_COMPLEMENTO:
    print("Rodando complementacao com run_id fixo:", RUN_ID_MADRUGADA)
    manifest_complemento_path = run_collector(
        RUN_ID_MADRUGADA,
        DATA_INICIO_COMPLEMENTO,
        DATA_FIM_COMPLEMENTO,
        sample=False,
        resume=True,
    )
    manifest_complemento = show_manifest(manifest_complemento_path)
else:
    print("Altere EXECUTAR_COMPLEMENTO para True quando quiser iniciar a complementacao completa.")

## Consultar complementacao em andamento

Use esta celula em outra sessao do Colab para consultar manifesto, autosave e ultimas linhas de log do `run_id` fixo.

In [ ]:
autosave_madrugada_path = DATA_ROOT / "manifests" / f"{RUN_ID_MADRUGADA}.autosave.json"
manifest_madrugada_path = DATA_ROOT / "manifests" / f"{RUN_ID_MADRUGADA}.json"

if manifest_madrugada_path.exists():
    show_manifest(manifest_madrugada_path)
elif autosave_madrugada_path.exists():
    show_manifest(autosave_madrugada_path)
else:
    print("Ainda nao ha manifest/autosave para", RUN_ID_MADRUGADA)

tail_log_for_run(RUN_ID_MADRUGADA, n=30)